# Experiment 5.0.1 — Three-way Analog-Head Control

Analysis-only notebook. Training is performed by a 9-task Slurm array. This notebook aggregates finalized results for three analog-head controls and compares them with historical Exp3.0.5 and Exp5.0 references.

The design separates three possible causes of the Exp3/Exp5 timestep-CE gap: protocol reproduction, hidden-neuron dynamics, and the spiking output path.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'notebooks').is_dir():
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_5_0_1_exp3_analog_head_control' / 'analog_head_three_way_control_v3'
runs = pd.read_csv(ART / 'runs.csv')
comparison = pd.read_csv(ART / 'comparison_runs.csv')
manifest = json.loads((ART / 'manifest.json').read_text(encoding='utf-8'))
display(manifest)
display(runs)

## New-control aggregate

Each condition uses seeds `(11, 23, 101)`. Mean, SD, and count are computed here from finalized per-seed rows.

In [ ]:
primary_columns = [
    'native_test_ba',
    'full_count_test_ba',
    'fixed250_ordered_test_ba',
    'relative10_ordered_test_ba',
]
new_summary = runs.groupby('condition')[primary_columns].agg(['mean', 'std', 'count'])
display(new_summary)

## Unified comparison table

`native_or_output_ba` is native analog-head segment BA for Exp3/new controls and Output WholeCount BA for the historical Exp5 spiking-output references. The L2 frozen-probe columns are the directly comparable representation diagnostics.

In [ ]:
metric_columns = [
    'native_or_output_ba',
    'full_count_ba',
    'fixed250_ordered_ba',
    'relative10_ordered_ba',
]
comparison_summary = (
    comparison.groupby(['source', 'variant'])[metric_columns]
    .agg(['mean', 'std', 'count'])
)
display(comparison_summary)

## Gate 1 — exact Exp3 reproduction

Compare `exp3_exact_analog` against historical Exp3.0.5 on the exact three seeds. Near-zero paired deltas are the prerequisite for any mechanistic interpretation.

In [ ]:
exp3_exact = comparison[(comparison['source'] == 'exp5_0_1_control') & (comparison['variant'] == 'exp3_exact_analog')].set_index('seed')
historical_exp3 = comparison[comparison['source'] == 'exp3_0_5_reference'].set_index('seed')
common_repro = sorted(set(exp3_exact.index) & set(historical_exp3.index))
paired_reproduction = pd.DataFrame(index=common_repro)
for metric in metric_columns:
    paired_reproduction[metric] = exp3_exact.loc[common_repro, metric] - historical_exp3.loc[common_repro, metric]
display(paired_reproduction)
display(pd.DataFrame({'mean_delta': paired_reproduction.mean(), 'sd_delta': paired_reproduction.std()}))

## Gate 2 — hidden-dynamics effect under the same Exp5 stream

`exp3_synaptic_exp5stream_analog` and `exp5_macro_exp5stream_analog` use the same Exp5 initialization stream, the same train/val/test loader streams, the same analog head, the same objective, and the same checkpoint rule. Their paired difference therefore isolates the hidden-neuron implementation as tightly as this experiment allows.

In [ ]:
syn_exp5stream = comparison[(comparison['source'] == 'exp5_0_1_control') & (comparison['variant'] == 'exp3_synaptic_exp5stream_analog')].set_index('seed')
macro_exp5stream = comparison[(comparison['source'] == 'exp5_0_1_control') & (comparison['variant'] == 'exp5_macro_exp5stream_analog')].set_index('seed')
common_hidden = sorted(set(syn_exp5stream.index) & set(macro_exp5stream.index))
paired_hidden = pd.DataFrame(index=common_hidden)
for metric in metric_columns:
    paired_hidden[metric] = macro_exp5stream.loc[common_hidden, metric] - syn_exp5stream.loc[common_hidden, metric]
display(paired_hidden)
display(pd.DataFrame({'mean_delta_macro_minus_synaptic': paired_hidden.mean(), 'sd_delta': paired_hidden.std()}))

## Gate 3 — analog head versus Exp5 spiking-output path

Compare `exp5_macro_exp5stream_analog` with historical Exp5 `timestep_ce + binary`. The hidden forward dynamics and Exp5 random/data stream are matched by construction, but only common seeds `11` and `23` exist in both experiments and are treated as paired. `multi_ho` is included as an output-capacity reference, not as an exact architecture pair.

In [ ]:
paired_exp5_tables = {}
for variant in ('binary', 'multi_ho'):
    ref = comparison[(comparison['source'] == 'exp5_0_spiking_output_reference') & (comparison['variant'] == variant)].set_index('seed')
    common_exp5 = sorted(set(macro_exp5stream.index) & set(ref.index))
    delta = pd.DataFrame(index=common_exp5)
    for metric in metric_columns:
        delta[metric] = macro_exp5stream.loc[common_exp5, metric] - ref.loc[common_exp5, metric]
    paired_exp5_tables[variant] = delta
    print(f'Exp5-Macro analog - Exp5 {variant}, paired seeds={common_exp5}')
    display(delta)
    display(pd.DataFrame({'mean_delta': delta.mean(), 'sd_delta': delta.std()}))

## Representation comparison plot

In [ ]:
plot_metrics = ['full_count_ba', 'fixed250_ordered_ba', 'relative10_ordered_ba']
labels = {
    'full_count_ba': 'L2 FullCount + Linear',
    'fixed250_ordered_ba': 'L2 Fixed250 + Linear',
    'relative10_ordered_ba': 'L2 Relative10 + Linear',
}
plot_frame = comparison.groupby(['source', 'variant'])[plot_metrics].mean().reset_index()
plot_frame['model'] = plot_frame['source'] + ' / ' + plot_frame['variant']
x = np.arange(len(plot_metrics))
width = 0.8 / len(plot_frame)
fig, ax = plt.subplots(figsize=(14, 6))
for i, row in plot_frame.iterrows():
    values = [100 * row[m] for m in plot_metrics]
    ax.bar(x + (i - (len(plot_frame)-1)/2) * width, values, width=width, label=row['model'])
ax.set_xticks(x, [labels[m] for m in plot_metrics])
ax.set_ylabel('Test balanced accuracy (%)')
ax.set_title('Exp5.0.1 three-way analog-head control vs Exp3 / Exp5 references')
ax.legend(fontsize=7)
fig.tight_layout()
plt.show()

## Interpretation logic

1. If `exp3_exact_analog` does not reproduce historical Exp3, stop and diagnose protocol drift.
2. If reproduction succeeds, use B vs C to measure the hidden implementation effect under the same Exp5 stream.
3. If B and C are close but C is much stronger than existing Exp5 binary, the spiking output path / its optimization and checkpoint regime is the main cause of the timestep-CE degradation.
4. If C is already much weaker than B, hidden dynamics/surrogate implementation contributes materially and the output LIF cannot be blamed alone.
5. Use Multi-HO only to judge whether higher instantaneous output capacity partially compensates for the spiking-output bottleneck.